# Nuage de points sur carte : éditions de Venise, Paris et Lyon

Pour ces trois villes d'édition des *Métamorphoses* d'Ovide, un nuage de points sur une
**vraie carte géographique** (comme `carte_circulation.ipynb`) plutôt qu'un graphique abstrait :
Venise, Paris et Lyon sont à leur position réelle, et autour de chaque ville un petit nuage de
points — un point par édition connue, disposé en spirale pour ne pas se chevaucher. La couleur
encode la technique de gravure (bois / cuivre), pour voir si une ville bascule plus tôt ou plus
tard vers le cuivre.

Même source de données que `carte_circulation.ipynb` : `retours_celine/BNU_corpus.ods`
(feuille `Synthèse`).

In [29]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "nuage_editions_venise_paris_lyon.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS cellule par cellule que pour la carte (`pandas.read_excel` masque des
colonnes de ce fichier — voir `carte_circulation.ipynb` pour le détail). On ne garde que les
éditions dont la ville normalisée est Lyon, Paris ou Venise.

In [30]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

# Seules les 3 villes qui nous intéressent ici (pas besoin de la table de correspondance
# complète de carte_circulation.ipynb, qui couvre 19 villes)
VILLES_RETENUES = {"Paris": "Paris", "[Paris]": "Paris", "Lyon": "Lyon", "Venise": "Venise"}
ORDRE_VILLES = ["Lyon", "Paris", "Venise"]

def categorie_technique(t):
    t = (t or "").strip().lower()
    if t == "bois":
        return "bois"
    if t == "cuivre":
        return "cuivre"
    return "inconnue"  # vide, "inaccessible", "?", etc. -> repli neutre plutôt que fragmenter

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

editions = []
for row in corpus:
    ville = VILLES_RETENUES.get(row.get("ville", "").strip())
    if ville is None:
        continue
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    editions.append({
        "ville": ville,
        "annee": annee,
        "titre": titre.strip(),
        "technique": categorie_technique(row.get("technique", "")),
        "graveur": row.get(COL_GRAVEUR, "").strip(),
        "lien": next((row[c] for c in ["version numérisée 1", "version numérisée 2",
                                        "Biblioteca Digital Ovidiana", "url catalogue"]
                      if row.get(c, "").strip().startswith("http")), None),
    })

print(len(editions), "éditions retenues (Lyon, Paris, Venise)")
for v in ORDRE_VILLES:
    sous = [e for e in editions if e["ville"] == v]
    print(f"  {v:8s} {len(sous):2d} éditions, {min(e['annee'] for e in sous)}–{max(e['annee'] for e in sous)}")

from collections import Counter
print("techniques :", Counter(e["technique"] for e in editions))

64 éditions retenues (Lyon, Paris, Venise)
  Lyon     21 éditions, 1510–1697
  Paris    25 éditions, 1493–1737
  Venise   18 éditions, 1497–1624
techniques : Counter({'bois': 43, 'cuivre': 18, 'inconnue': 3})


## 2. Choix de visualisation

- **Forme** : les deux à la fois, combinées plutôt que l'une remplaçant l'autre. Pour chaque
  ville, un panneau avec (a) une **carte géographique réelle** (position vraie, nuage en
  spirale de phyllotaxie autour) et (b) juste en dessous une **frise chronologique** compacte
  (axe = année 1490-1750) pour la même ville — la carte donne le lieu, la frise donne le
  moment, les deux se lisent ensemble sans se remplacer.
- **Couleur** : réservée à la technique de gravure (bois / cuivre) dans les deux
  représentations, mêmes couleurs catégorielles validées (bleu `#2a78d6` bois, aqua `#1baf7a`
  cuivre, gris neutre `#9a9993` pour technique inconnue) — cohérence entre carte et frise.
- **Validation de palette** : reportée en Python faute de `node` ; passe partout sauf un WARN
  sur le contraste de l'aqua en clair, mitigé par un contour foncé sur les points.
- **Interaction** : infobulle au survol de chaque point, sur la carte comme sur la frise
  (même contenu : titre, année, technique, graveur, lien), légende toujours visible, et vue
  tableau dépliable (accessibilité).
- **Mode sombre** : mêmes jeux de couleurs validés séparément que la version précédente.

In [31]:
import math

COULEURS_CLAIR = {"bois": "#2a78d6", "cuivre": "#1baf7a", "inconnue": "#9a9993"}
COULEURS_SOMBRE = {"bois": "#3987e5", "cuivre": "#199e70", "inconnue": "#8a8983"}
LIBELLES_TECHNIQUE = {"bois": "Bois", "cuivre": "Cuivre", "inconnue": "Technique inconnue"}

# Mêmes coordonnées que carte_circulation.ipynb
VILLES_COORDS = {"Lyon": (45.764, 4.8357), "Paris": (48.8566, 2.3522), "Venise": (45.4408, 12.3155)}

RAYON_MAX_KM = 22            # étalement maximal du nuage autour de chaque ville
ANGLE_OR = math.radians(137.508)  # angle d'or : répartition régulière en spirale, sans grille

def decalage_spirale(i, n, rayon_max_km):
    """Spirale de phyllotaxie (rayon ∝ √i, angle = i × angle d'or) : répartit n'importe quel
    nombre de points régulièrement autour d'un centre, sans qu'ils se chevauchent."""
    if n <= 1:
        return 0.0, 0.0
    rayon = rayon_max_km * math.sqrt(i / (n - 1))
    angle = i * ANGLE_OR
    return rayon * math.cos(angle), rayon * math.sin(angle)

def deplacer_latlon(lat, lon, dx_km, dy_km):
    """Déplace un point de (dx_km vers l'est, dy_km vers le nord) en degrés lat/lon."""
    dlat = dy_km / 111.0
    dlon = dx_km / (111.0 * math.cos(math.radians(lat)))
    return lat + dlat, lon + dlon

# --- Frise chronologique compacte (une par ville, à droite de sa carte) ---
# Plus haute que la première version : les éditions proches dans le temps ont besoin de
# place pour être clairement séparées verticalement (voir plus bas).
FRISE_LARGEUR, FRISE_HAUTEUR = 320, 130
FRISE_MARGE = {"gauche": 12, "droite": 16, "haut": 18, "bas": 26}
FRISE_ANNEE_MIN, FRISE_ANNEE_MAX = 1490, 1750
FRISE_HAUTEUR_PLOT = FRISE_HAUTEUR - FRISE_MARGE["haut"] - FRISE_MARGE["bas"]
FRISE_Y_CENTRE = FRISE_MARGE["haut"] + FRISE_HAUTEUR_PLOT / 2

def frise_x(annee):
    t = (annee - FRISE_ANNEE_MIN) / (FRISE_ANNEE_MAX - FRISE_ANNEE_MIN)
    return FRISE_MARGE["gauche"] + t * (FRISE_LARGEUR - FRISE_MARGE["gauche"] - FRISE_MARGE["droite"])

# Deux points dans la même case de cette largeur (en pixels) sont jugés "trop proches
# horizontalement" et regroupés pour l'étalement vertical. Des cases de largeur fixe
# plutôt qu'un chaînage de proche en proche : le chaînage a tendance à faire boule de neige
# (si A est proche de B et B proche de C, A et C finissent regroupés même s'ils sont loin
# l'un de l'autre), ce qui étalait presque toutes les éditions d'une ville sur toute la
# hauteur au lieu de ne séparer que les vraies coïncidences locales.
LARGEUR_CASE_PX = 16

groupes = {}
for e in editions:
    groupes.setdefault(e["ville"], []).append(e)

points = []
for ville, groupe in groupes.items():
    groupe_trie = sorted(groupe, key=lambda e: e["annee"])
    n = len(groupe_trie)
    lat0, lon0 = VILLES_COORDS[ville]

    # Regroupe par case fixe le long de l'axe des années (pas par année exacte, ni par
    # chaînage de proche en proche)
    cases = {}
    for idx, e in enumerate(groupe_trie):
        x = frise_x(e["annee"])
        case = int((x - FRISE_MARGE["gauche"]) // LARGEUR_CASE_PX)
        cases.setdefault(case, []).append(idx)
    position_dans_case = {}
    for indices in cases.values():
        for j, idx in enumerate(indices):
            position_dans_case[idx] = (j, len(indices))

    for i, e in enumerate(groupe_trie):
        # -- position géographique (spirale autour de la ville) --
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)

        # -- position sur la frise chronologique : les éditions tombant dans la même case
        #    (même année ou années très voisines) sont nettement séparées verticalement
        #    (jusqu'à 25px, largement plus que le rayon des points) --
        j, m = position_dans_case[i]
        espacement_f = min(25, (FRISE_HAUTEUR_PLOT * 0.85) / max(m - 1, 1)) if m > 1 else 0
        fy = FRISE_Y_CENTRE + (j - (m - 1) / 2) * espacement_f
        fx = frise_x(e["annee"])

        points.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "fx": round(fx, 1),
            "fy": round(fy, 1),
            "ville": ville,
            "annee": e["annee"],
            "titre": e["titre"],
            "technique": e["technique"],
            "graveur": e["graveur"],
            "lien": e["lien"],
        })

print(len(points), "points positionnés (carte en spirale + frise chronologique)")

64 points positionnés (carte en spirale + frise chronologique)


## 3. Génération de la carte + frise (HTML autonome)

Mise en page en **deux colonnes** : à gauche, les trois cartes empilées verticalement (une
par ville) ; à droite, les trois frises chronologiques empilées, chacune alignée sur la même
ligne que sa carte.

In [32]:
def ligne_tableau(p):
    lien_html = f'<a href="{p["lien"]}" target="_blank">voir</a>' if p["lien"] else ""
    return (
        f'<tr><td>{p["annee"]}</td><td>{p["titre"]}</td>'
        f'<td>{LIBELLES_TECHNIQUE[p["technique"]]}</td><td>{p["graveur"]}</td>'
        f'<td>{lien_html}</td></tr>'
    )

def tableau_ville(ville):
    lignes = "\n".join(
        ligne_tableau(p) for p in sorted(points, key=lambda p: p["annee"]) if p["ville"] == ville
    )
    return f'''<table class="tableau-detaille" id="tableau-{ville}">
    <thead><tr><th>Année</th><th>Titre</th><th>Technique</th><th>Graveur</th><th>Lien</th></tr></thead>
    <tbody>
      {lignes}
    </tbody>
  </table>'''

# --- Frise chronologique : axe (partagé, même échelle pour les 3 villes) + points par ville ---
FRISE_TICKS = [1500, 1600, 1700]

def frise_axes_svg():
    x_debut, x_fin = frise_x(FRISE_ANNEE_MIN), frise_x(FRISE_ANNEE_MAX)
    elements = [
        # ligne de base : l'axe du temps lui-même, du début à la fin de la période
        f'<line x1="{x_debut:.1f}" y1="{FRISE_Y_CENTRE:.1f}" x2="{x_fin:.1f}" y2="{FRISE_Y_CENTRE:.1f}" class="ligne-base"/>',
        # petite pointe de flèche à droite : le sens du temps
        f'<path d="M {x_fin - 1:.1f} {FRISE_Y_CENTRE - 4:.1f} L {x_fin + 6:.1f} {FRISE_Y_CENTRE:.1f} '
        f'L {x_fin - 1:.1f} {FRISE_Y_CENTRE + 4:.1f} Z" class="pointe-base"/>',
    ]
    for t in FRISE_TICKS:
        x = frise_x(t)
        elements.append(
            f'<line x1="{x:.1f}" y1="{FRISE_MARGE["haut"]}" x2="{x:.1f}" '
            f'y2="{FRISE_MARGE["haut"] + FRISE_HAUTEUR_PLOT:.1f}" class="grille-frise"/>'
        )
        elements.append(f'<line x1="{x:.1f}" y1="{FRISE_Y_CENTRE-4:.1f}" x2="{x:.1f}" y2="{FRISE_Y_CENTRE+4:.1f}" class="tick-frise"/>')
        elements.append(
            f'<text x="{x:.1f}" y="{FRISE_MARGE["haut"] + FRISE_HAUTEUR_PLOT + 16:.1f}" '
            f'text-anchor="middle" class="etiquette-annee-frise">{t}</text>'
        )
    return "\n".join(elements)

FRISE_AXES = frise_axes_svg()

def frise_cercles_ville(ville):
    cercles = []
    for i, p in enumerate(points):
        if p["ville"] != ville:
            continue
        cercles.append(
            f'<circle class="point-frise" data-i="{i}" cx="{p["fx"]}" cy="{p["fy"]}" r="5" '
            f'fill="var(--c-{p["technique"]})"/>'
        )
    return "\n".join(cercles)

# Deux colonnes (carte à gauche, frise à droite) : chaque ville ajoute au flux, dans l'ordre,
# [carte, frise, bouton (pleine largeur), tableau (pleine largeur)] ; avec 2 colonnes de
# grille, le bouton et le tableau — qui s'étendent sur les 2 colonnes — retombent
# naturellement sur leur propre ligne juste sous la paire carte/frise de leur ville.
cellules = []
for ville in VILLES_COORDS:
    cellules.append(f'<div class="cellule-carte"><h2>{ville}</h2><div class="carte" id="carte-{ville}"></div></div>')
    cellules.append(
        f'<div class="panneau-frise"><svg class="frise" viewBox="0 0 {FRISE_LARGEUR} {FRISE_HAUTEUR}">\n'
        f'    {FRISE_AXES}\n'
        f'    {frise_cercles_ville(ville)}\n'
        f'  </svg></div>'
    )
    cellules.append(f'<button class="bascule action-ville" data-ville="{ville}">Afficher le tableau détaillé</button>')
    cellules.append(tableau_ville(ville))
blocs_carte = "\n".join(cellules)

TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Nuage de points : éditions de Venise, Paris et Lyon</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23;
    --c-bois: #2a78d6; --c-cuivre: #1baf7a; --c-inconnue: #9a9993;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2;
      --c-bois: #3987e5; --c-cuivre: #199e70; --c-inconnue: #8a8983;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1100px; margin:0 auto; padding:16px 20px 32px; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 14px; }
  .legende { display:flex; gap:18px; flex-wrap:wrap; font-size:12px; margin:0 0 6px; }
  .legende .item { display:flex; align-items:center; gap:6px; }
  .legende .puce { width:11px; height:11px; border-radius:50%; border:1px solid var(--contour-point); }

  /* Colonne de gauche = cartes, colonne de droite = frises ; le bouton et le tableau de
     chaque ville s'étendent sur les 2 colonnes, juste sous sa paire carte/frise. */
  .rangee-cartes { display:grid; grid-template-columns:3fr 2fr; align-items:center;
    gap:8px 20px; margin-top:14px; }
  @media (max-width:760px) { .rangee-cartes { grid-template-columns:1fr; } }
  .cellule-carte h2 { font-size:15px; margin:0 0 6px; color:var(--texte-fort); }
  .carte { height:300px; border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.25); }

  .panneau-frise { background:var(--surface); border:1px solid var(--trait); border-radius:6px;
    box-shadow:0 1px 6px rgba(0,0,0,.15); padding:6px 6px 2px; }
  .frise { width:100%; height:auto; overflow:visible; display:block; }
  .ligne-base { stroke:var(--texte-att); stroke-width:1.5; }
  .pointe-base { fill:var(--texte-att); }
  .grille-frise { stroke:var(--trait); stroke-width:1; }
  .tick-frise { stroke:var(--texte-att); stroke-width:1; }
  .etiquette-annee-frise { font-size:10px; fill:var(--texte-att); }
  .point-frise { stroke:var(--contour-point); stroke-width:1.2; fill-opacity:.9; cursor:pointer;
    transition:r .15s; filter:drop-shadow(0 1px 1px rgba(0,0,0,.35)); }
  .point-frise:hover, .point-frise.actif { r:8; fill-opacity:1; }

  .action-ville { grid-column:1 / -1; justify-self:start; margin:2px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { grid-column:1 / -1; width:100%; border-collapse:collapse;
    font-size:12px; margin:2px 0 6px; display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-bois); }

  .leaflet-tooltip { font-family:Georgia,serif; font-size:12px; background:var(--surface);
    border:1px solid var(--contour-point); color:var(--texte-fort); padding:4px 9px;
    box-shadow:0 1px 5px rgba(0,0,0,.3); }
  .infobulle { position:absolute; pointer-events:none; background:var(--surface);
    border:1px solid var(--contour-point); border-radius:5px; padding:6px 10px; font-size:12px;
    max-width:260px; opacity:0; transition:opacity .1s; box-shadow:0 2px 8px rgba(0,0,0,.3); z-index:2000; }
</style></head><body>
<div class="page">
  <h1>Éditions de Venise, Paris et Lyon</h1>
  <p class="souschapo">Un point = une édition connue, disposée en spirale autour de sa ville
    sur la carte (à gauche), et par année sur sa frise (à droite). Couleur = technique de gravure.</p>
  <div class="legende" id="legende"></div>
  <div class="rangee-cartes">
    __BLOCS_CARTE__
  </div>
  <div class="infobulle" id="infobulle"></div>
</div>
<script>
  const points = __POINTS__;
  const villesCoords = __VILLES_COORDS__;
  const libellesTechnique = __LIBELLES__;
  const couleursClair = __COULEURS_CLAIR__;

  // Légende (une puce par catégorie de technique réellement présente dans les données)
  const techniquesPresentes = [...new Set(points.map(p => p.technique))]
    .sort((a, b) => (a === "inconnue") - (b === "inconnue"));  // "inconnue" en dernier
  document.getElementById('legende').innerHTML = techniquesPresentes.map(t =>
    '<div class="item"><span class="puce" style="background:' + couleursClair[t] + '"></span>' +
    libellesTechnique[t] + '</div>'
  ).join('');

  function contenuInfobulle(p) {
    const lien = p.lien ? '<br><a href="' + p.lien + '" target="_blank">→ voir</a>' : '';
    return '<b>' + p.titre + '</b><br>' +
      '<span>' + p.ville + ', ' + p.annee + ' · ' + libellesTechnique[p.technique] + '</span>' +
      (p.graveur ? '<br>' + p.graveur : '') + lien;
  }

  // Une carte Leaflet par ville, chacune cadrée sur son propre nuage (pas sur les 3 villes
  // à la fois : Lyon/Paris/Venise sont loin les unes des autres, une seule carte les
  // englobant montrerait surtout du vide entre elles).
  for (const ville of Object.keys(villesCoords)) {
    const pointsVille = points.filter(p => p.ville === ville);
    const map = L.map('carte-' + ville, {scrollWheelZoom:false});
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
      {attribution:'© OpenStreetMap contributors', maxZoom:18}).addTo(map);

    const bornes = L.latLngBounds(pointsVille.map(p => [p.lat, p.lon]));
    map.fitBounds(bornes, {padding:[36, 36]});

    pointsVille.forEach(p => {
      L.circleMarker([p.lat, p.lon], {
        radius:7, color:'#3e2c23', weight:1, fillColor: couleursClair[p.technique], fillOpacity:.85
      }).addTo(map).bindTooltip(contenuInfobulle(p), {sticky:true, maxWidth:260});
    });
  }

  // Infobulle partagée pour les points de la frise chronologique (SVG, à droite de chaque carte)
  const infobulle = document.getElementById('infobulle');
  const page = document.querySelector('.page');
  document.querySelectorAll('.point-frise').forEach(cercle => {
    const p = points[+cercle.dataset.i];
    cercle.addEventListener('mouseenter', () => {
      cercle.classList.add('actif');
      infobulle.innerHTML = contenuInfobulle(p);
      infobulle.style.opacity = 1;
    });
    cercle.addEventListener('mousemove', (ev) => {
      const r = page.getBoundingClientRect();
      infobulle.style.left = (ev.clientX - r.left + 14) + 'px';
      infobulle.style.top = (ev.clientY - r.top + 14) + 'px';
    });
    cercle.addEventListener('mouseleave', () => {
      cercle.classList.remove('actif');
      infobulle.style.opacity = 0;
    });
  });

  // Un bouton "Afficher le tableau détaillé" par ville, juste sous sa carte/frise.
  document.querySelectorAll('.action-ville').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const tableau = document.getElementById('tableau-' + bouton.dataset.ville);
      const visible = tableau.classList.toggle('visible');
      bouton.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
    });
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__BLOCS_CARTE__", blocs_carte)
    .replace("__POINTS__", json.dumps(points, ensure_ascii=False))
    .replace("__VILLES_COORDS__", json.dumps(VILLES_COORDS, ensure_ascii=False))
    .replace("__LIBELLES__", json.dumps(LIBELLES_TECHNIQUE, ensure_ascii=False))
    .replace("__COULEURS_CLAIR__", json.dumps(COULEURS_CLAIR, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Carte + frise écrites dans", CHEMIN_SORTIE)

Carte + frise écrites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/nuage_editions_venise_paris_lyon.html
